# Data Processing


In [ ]:
# Load csv as pandas dataframe
import pandas as pd

# Load csv as pandas dataframe
cleanedPatternDf = pd.read_csv('Datasets/scraped_blog_tables.csv')

cleanedPatternDf['Start'] = pd.to_datetime(cleanedPatternDf['Start'])
cleanedPatternDf['End'] = pd.to_datetime(cleanedPatternDf['End'])

cleanedPatternDf

## Remove data of missing data Symbols


In [ ]:
# get the csv file names exept the extension in the folder OHLC data 
import os
from os import listdir
from os.path import isfile, join
onlyfiles = [f for f in listdir('Datasets/OHLC data') if isfile(join('Datasets/OHLC data', f))]
onlyfiles = [f.split('.')[0] for f in onlyfiles]
print(onlyfiles)

#  now remove any row that contain a Symbol that is not in the list of csv file names from the filteredPatternDf dataframe
cleanedPatternDf = cleanedPatternDf[cleanedPatternDf['Symbol'].isin(onlyfiles)]
cleanedPatternDf.head()

## Filter for the selected patterns


In [ ]:
from utils.formatAndPreprocessNewPatterns import filter_to_get_selected_patterns

filteredPatternDf = filter_to_get_selected_patterns(cleanedPatternDf)

# get the list of unique Chart Patterns in Chart Pattern column
uniqueChartPatterns = filteredPatternDf['Chart Pattern'].unique()

# make new directory to save the filtered data 
if not os.path.exists('Datasets/New_pattern_set_data'):
    os.makedirs('Datasets/New_pattern_set_data')

filteredPatternDf.to_csv('Datasets/New_pattern_set_data/filteredPatternDf.csv', index=False)

print(uniqueChartPatterns)
filteredPatternDf 

In [ ]:
filteredPatternDf_un_aug = filteredPatternDf.copy()

## Width Augment


In [ ]:
import numpy as np
from utils.formatAndPreprocessNewPatterns import width_augmentation

min_aug_len = 3
aug_len_fraction = 0.5
np.random.seed(69)

filteredPatternDf_w_aug = width_augmentation(filteredPatternDf_un_aug, min_aug_len, aug_len_fraction, make_duplicates = False, keep_original = False)

## Width augment data switch


In [ ]:
filteredPatternDf = filteredPatternDf_un_aug
# filteredPatternDf =filteredPatternDf_w_aug

## Add No Pattern Data


In [ ]:
oob_no_pattern_df = pd.read_csv('Datasets/VanilaDataset/oob_no_pattern_stats_df.csv', index_col=0)
oob_no_pattern_df["Avg_Prob"] = oob_no_pattern_df['oob_sum'] / oob_no_pattern_df['oob_count']
sorted_df = oob_no_pattern_df.sort_values(by='Avg_Prob', ascending=False)
top_500_instances = sorted_df.head(1000)
no_pattern_all_df = pd.read_csv('Datasets/VanilaDataset/no_pattern_10000_df.csv', index_col=0)
# Read the CSV file
instance_index_mapping_df = pd.read_csv('Datasets/VanilaDataset/instance_index_mapping_df.csv', index_col=0)
# Convert back to dictionary
instance_index_mapping = dict(zip(instance_index_mapping_df['Instance'], instance_index_mapping_df['Index']))

In [ ]:
top_500_instances = top_500_instances.index

no_pattern_df =no_pattern_all_df.loc[top_500_instances]
no_pattern_df

In [ ]:
# concatenate the two dataframes
final_pattern_df = pd.concat([filteredPatternDf, no_pattern_df], ignore_index=True)
final_pattern_df_w_aug =  pd.concat([filteredPatternDf_w_aug, no_pattern_df], ignore_index=True)

final_pattern_df.to_csv('Datasets/New_pattern_set_data/final_pattern_df.csv', index=True)
final_pattern_df_w_aug.to_csv('Datasets/New_pattern_set_data/final_pattern_df_w_aug.csv', index=True)

# convert Start and End columns to datetime
final_pattern_df['Start'] = pd.to_datetime(final_pattern_df['Start'])
final_pattern_df['End'] = pd.to_datetime(final_pattern_df['End'])

final_pattern_df_w_aug['Start'] = pd.to_datetime(final_pattern_df_w_aug['Start'])
final_pattern_df_w_aug['End'] = pd.to_datetime(final_pattern_df_w_aug['End'])

final_pattern_df

In [ ]:
# get the number of each chart pattern
pattern_counts = final_pattern_df['Chart Pattern'].value_counts()
# Print the counts
print(pattern_counts)

### Test Train Split


In [ ]:
# split the data into train and test set with 80% of the data in the train set
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(final_pattern_df, test_size=0.2, random_state=42)

train_df_w_aug, test_df_w_aug = train_test_split(final_pattern_df_w_aug, test_size=0.2, random_state=42)

In [ ]:
from utils.formatAndPreprocessNewPatterns import dataset_format

train_df_ohlc = dataset_format(train_df)
test_df_ohlc = dataset_format(test_df)

train_df_w_aug_ohlc = dataset_format(train_df_w_aug)
test_df_w_aug_ohlc = dataset_format(test_df_w_aug)

In [ ]:
train_df_ohlc

In [ ]:
# save the train and test dataframes to csv files
train_df_ohlc.to_csv('Datasets/New_pattern_set_data/train_df_ohlc.csv', index=True)
test_df_ohlc.to_csv('Datasets/New_pattern_set_data/test_df_ohlc.csv', index=True)

train_df_w_aug_ohlc.to_csv('Datasets/New_pattern_set_data/train_df_w_aug_ohlc.csv', index=True)
test_df_w_aug_ohlc.to_csv('Datasets/New_pattern_set_data/test_df_w_aug_ohlc.csv', index=True)

train_df.to_csv('Datasets/New_pattern_set_data/train_df.csv', index=True)
test_df.to_csv('Datasets/New_pattern_set_data/test_df.csv', index=True)

train_df_w_aug.to_csv('Datasets/New_pattern_set_data/train_df_w_aug.csv', index=True)
test_df_w_aug.to_csv('Datasets/New_pattern_set_data/test_df_w_aug.csv', index=True)

## Train data aug


In [ ]:
import numpy as np
import pandas as pd
import mplfinance as mpf
import matplotlib.pyplot as plt
from utils.formatAndPreprocessNewPatterns import get_patetrn_name_by_encoding



# Define a linear trend function (y = a*x + b)
def linear_trend(x, slope=0.1, intercept=0):
    return slope * x + intercept

# Define a non-linear (quadratic) trend function (y = a*x^2 + b*x + c)
def quadratic_trend(x, a=0.001, b=0.01, c=0):
    return a * x**2 + b * x + c

# Define a sine wave trend function
def sine_trend(x, amplitude=0.02, frequency=0.05 ):
    return amplitude * np.sin(frequency * x)

def plot_trend_function(x_values, trend, trend_function_name):
    """Plot the trend function"""
    plt.figure(figsize=(3, 2))
    plt.plot(x_values, trend, label=f'{trend_function_name}')
    plt.title('Trend Function Plot')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.show()
    
def plot_original_and_augmented_data(original_data, augmented_data):
    """Plot original and augmented data as candlestick charts"""
    # Prepare original and noisy data for plotting (OHLC only)
    original_data_for_plot = original_data[['Open', 'High', 'Low', 'Close']].reset_index(drop=True)
    augmented_data_for_plot = augmented_data[['Open', 'High', 'Low', 'Close']].reset_index(drop=True)

    # Create two subplots to compare original and noisy data
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Create a DatetimeIndex for the plots (ensure it matches the number of rows in the data)
    date_range = pd.date_range(start='2024-01-01', periods=len(original_data_for_plot), freq='D')

    # Assign the index to the original and noisy data for plotting
    original_data_for_plot.index = date_range
    augmented_data_for_plot.index = date_range

    # Plot the original data
    mpf.plot(original_data_for_plot, type='candle', ax=axes[0], style='yahoo', volume=False)
    axes[0].grid(True)  # Enable gridlines
    axes[0].set_title("Original OHLC Candlestick Chart")

    # Plot the noisy data (augmented with trend)
    mpf.plot(augmented_data_for_plot, type='candle', ax=axes[1], style='yahoo', volume=False)
    axes[1].grid(True)  # Enable gridlines
    axes[1].set_title("Augmented OHLC Candlestick Chart with Trend")

    # Show the comparison plots
    plt.tight_layout()
    plt.show()

def traindata_augment(Dataset):
    # Define the number of samples to generate
    n_total_samples = 1000
    # Create a counter for generating new first-level indices
    new_first_level_counter = Dataset.index.get_level_values(0).max() + 1

    # Loop through the unique chart patterns in the Dataset
    for pattern in Dataset['Pattern'].unique():
        # Filter the Dataset for the current pattern
        pattern_data = Dataset[Dataset['Pattern'] == pattern]

        # Get the unique values from the first level of the multi-index
        unique_first_level_index = pattern_data.index.get_level_values(0).unique()
        
        # get two random samples from the unique first level index
        instances_to_plot = np.random.choice(unique_first_level_index, 2, replace=False)

        # Set the number of augmented samples to create
        n_samples = n_total_samples - len(unique_first_level_index)

        # Loop through the number of samples to generate
        for i in range(n_samples):
            # Randomly select a section of the DataFrame based on the first-level index
            random_first_level_value = np.random.choice(unique_first_level_index)
            random_section = pattern_data.loc[(random_first_level_value, slice(None)), :]

            # Get the difference between max and min of Close column
            diff = random_section['Close'].max() - random_section['Close'].min()
            adjusted_diff = np.log(1 + diff)  # Adding 1 to avoid log(0)

            # Introduce randomness into the noise level
            noise_level = adjusted_diff * (0.08 + np.random.uniform(-0.01, 0.01))  # Adding random factor
            sub_noise_level = adjusted_diff * (0.01 + np.random.uniform(-0.004, 0.004))  # Adding random factor

            x_values = np.arange(len(random_section))

            # Randomly select a trend function
            trend_function = np.random.choice([linear_trend, quadratic_trend, sine_trend])

            # Modify the trend function parameters based on price range and random factors
            if trend_function == linear_trend:
                slope = np.random.uniform(0.005, 0.05) * adjusted_diff * (0.4 + np.random.uniform(-0.01, 0.01))   # Slope depends on price difference
                trend = linear_trend(x_values, slope=slope)
                if (random_first_level_value in instances_to_plot) :
                    print('slope:', slope)
            elif trend_function == quadratic_trend:
                a = np.random.uniform(0.00001, 0.0005) * adjusted_diff * (0.4 + np.random.uniform(-0.01, 0.01))   # Quadratic coefficient scaled by diff
                b = np.random.uniform(0.001, 0.01) * adjusted_diff
                trend = quadratic_trend(x_values, a=a, b=b)
                if (random_first_level_value in instances_to_plot) :
                    print('a:', a, 'b:', b)
            else:  # sine_trend
                amplitude = np.random.uniform(0.05, 1.2) * adjusted_diff * (0.4 + np.random.uniform(-0.01, 0.01))   # Amplitude depends on price difference
                frequency = np.random.uniform(0.01, 1)
                trend = sine_trend(x_values, amplitude=amplitude, frequency=frequency)
                if (random_first_level_value in instances_to_plot) :
                    print('amplitude:', amplitude, 'frequency:', frequency)



            # Add random noise (minor noise level) for variation in individual OHLC points
            noise = np.random.normal(0, sub_noise_level, random_section[['Open', 'High', 'Low', 'Close', 'Volume']].shape)
            noisy_data = random_section[['Open', 'High', 'Low', 'Close', 'Volume']] + noise

            # Add consistent noise across the same row (major noise level)
            row_noise = np.random.normal(0, noise_level, random_section[['Open']].shape)
            noisy_data['Open'] = random_section['Open'] + row_noise.squeeze()
            noisy_data['High'] = random_section['High'] + row_noise.squeeze()
            noisy_data['Low'] = random_section['Low'] + row_noise.squeeze()
            noisy_data['Close'] = random_section['Close'] + row_noise.squeeze()
            # noisy_data['Adj Close'] = random_section['Adj Close'] + row_noise.squeeze()
            noisy_data['Volume'] = random_section['Volume'] + row_noise.squeeze()

            # Add the trend equally to all OHLC columns
            noisy_data['Open'] += trend
            noisy_data['High'] += trend
            noisy_data['Low'] += trend
            noisy_data['Close'] += trend
            # noisy_data['Adj Close'] += trend
            noisy_data['Volume'] += trend  # You can adjust the impact of the trend on Volume if needed

            # Assign new first-level index to the noisy_data
            new_first_level_index = pd.MultiIndex.from_product([[new_first_level_counter], random_section.index.get_level_values(1)], names=['Index', 'Date'])
            noisy_data.index = new_first_level_index

            # Increment the first-level counter for the next sample
            new_first_level_counter += 1

            # Visualize the original and noisy data using candlestick charts
            if (random_first_level_value in instances_to_plot) :
                print(f"\nSelected Chart Pattern: {get_patetrn_name_by_encoding(pattern)}")
                print(f"\nSelected Trend Function: {trend_function.__name__}")
                plot_trend_function(x_values, trend, trend_function.__name__)
                plot_original_and_augmented_data(random_section, noisy_data)
                # pop the instance from the instances_to_plot
                instances_to_plot = instances_to_plot[instances_to_plot != random_first_level_value]
            
            # add Pattern column to the noisy data
            noisy_data['Pattern'] = pattern

            # Concatenate the noisy data to the original Dataset
            Dataset = pd.concat([Dataset, noisy_data], axis=0)

    return Dataset



In [ ]:
train_df_ohlc_aug = traindata_augment(train_df_ohlc)

train_df_w_aug_ohlc_aug = traindata_augment(train_df_w_aug_ohlc)

In [ ]:
# get the number of unique instences of level 0 index and Pattern combinations
for pattern in train_df_ohlc_aug['Pattern'].unique():
    # Filter the Dataset for the current pattern
    pattern_data = train_df_ohlc_aug[train_df_ohlc_aug['Pattern'] == pattern]

    # Get the unique values from the first level of the multi-index
    unique_first_level_index = pattern_data.index.get_level_values(0).unique()
    print(f"Pattern: {get_patetrn_name_by_encoding(pattern)} - Unique Instances: {len(unique_first_level_index)}")

### TrainAugSwitch


In [ ]:
train_df_ohlc_un_aug = train_df_ohlc.copy()
train_df_ohlc = train_df_ohlc_aug
# train_df_ohlc = train_df_ohlc_un_aug

## Length Normalize


In [ ]:
from utils.drawPlots import plot_ohlc_segment
from utils.formatAndPreprocessNewPatterns import normalize_ohlc_len

In [ ]:
target_len =30
train_df_ohlc_len_normed = normalize_ohlc_len(train_df_ohlc, target_len=target_len,plot_count=2)
test_df_ohlc_len_normed = normalize_ohlc_len(test_df_ohlc, target_len=target_len,plot_count=2)

#### Len Normed Width Auged


In [ ]:
train_df_ohlc_w_aug_len_normed = normalize_ohlc_len(train_df_w_aug_ohlc, target_len=target_len,plot_count=2)
test_df_ohlc_w_aug_len_normed = normalize_ohlc_len(test_df_w_aug_ohlc, target_len=target_len,plot_count=2)


## Data set type dict


In [ ]:
Datasets={}
Datasets["Plane OHLC"] = [train_df_ohlc, test_df_ohlc]
Datasets["Plane OHLC + traindata Aug"] = [train_df_ohlc_aug, test_df_ohlc]
Datasets["Width Aug OHLC"] = [train_df_w_aug_ohlc, test_df_w_aug_ohlc]
Datasets["Width Aug OHLC + traindata Aug"] = [train_df_w_aug_ohlc_aug , test_df_w_aug_ohlc]
Datasets["Length Normed OHLC"] = [train_df_ohlc_len_normed, test_df_ohlc_len_normed]
Datasets["Length Normed + Width Aug OHLC"] = [train_df_ohlc_w_aug_len_normed, test_df_ohlc_w_aug_len_normed]

# Classification


In [ ]:
model_stats = {}
models = {}

## Usual Classifier


In [ ]:
# # Define features and target
# features = ['Open', 'High', 'Low', 'Close', 'Volume']
# target = 'Pattern'
# series_length = 100  # Target series length for ROCKET

# def adjust_series_length(group, target_length):

#     series = group.values
#     current_length = len(series)

#     if current_length > target_length:
#         return series[:target_length]
#     else:
#         # Padding with zeros if shorter
#         padding = np.zeros((target_length - current_length, series.shape[1]))
#         return np.vstack([series, padding])


# def prepare_rocket_data(dataset, features, target, series_length):
#     # Group by 'Instance' and adjust each series length
#     adjusted = dataset.groupby(level=0).apply(
#         lambda group: adjust_series_length(group[features], series_length)
#     )
    
#     # Stack adjusted arrays
#     X = np.stack(adjusted.values)  # Shape: (num_samples, series_length, num_features)

#     # Extract targets (one per instance)
#     y = dataset.groupby(level=0)[target].first().values  # Shape: (num_samples,)
    
#     X = np.transpose(X, (0, 2, 1)) # shape: (num_samples, num_features, series_length)

#     return X, y

# # Prepare training and testing data
# X_train, y_train = prepare_rocket_data(train_df_ohlc, features, target, series_length)
# X_test, y_test = prepare_rocket_data(test_df_ohlc, features, target, series_length)

# print(f"X_train shape: {X_train.shape}")  # Expect (7080, 5, 100)
# print(f"y_train shape: {y_train.shape}")  # Expect (7080,)

### Rocket Classifier


In [ ]:
from sktime.transformations.panel.rocket import Rocket
from sklearn.pipeline import make_pipeline
from utils.FixedLengthTransformer import FixedLengthTransformer
import time
from xgboost import XGBClassifier


fl = FixedLengthTransformer(fixed_length=50, fill_value=0)
rocket = Rocket(num_kernels=10000)
xgbr = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='mlogloss', 
    n_estimators=100,
    # device="cuda"
)

clf_xgb = make_pipeline(
    fl,
    rocket,
    xgbr
)

models['rocket_xgb'] = clf_xgb



### Mini Rocket Classifier


In [ ]:
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.pipeline import make_pipeline
from utils.FixedLengthTransformer import FixedLengthTransformer
import time
from xgboost import XGBClassifier


fl = FixedLengthTransformer(fixed_length=50, fill_value=0)
mini_rocket = MiniRocketMultivariate(num_kernels=5000)
xgb_mr = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='mlogloss', 
    n_estimators=100,
    # device="cuda"
)

clf_xgb_mr = make_pipeline(
    fl,
    mini_rocket,
    xgb_mr
)


models['mini_rocket_xgb'] = clf_xgb_mr



### Multi Rocket Classifier


In [ ]:
from sktime.transformations.panel.rocket import MultiRocketMultivariate
from sklearn.pipeline import make_pipeline
from utils.FixedLengthTransformer import FixedLengthTransformer
import time
from xgboost import XGBClassifier


fl = FixedLengthTransformer(fixed_length=50, fill_value=0)
multi_rocket = MultiRocketMultivariate(num_kernels=5000)
xgb_mur = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='mlogloss', 
    n_estimators=100,
    # device="cuda"
)


clf_xgb_mur = make_pipeline(
    fl,
    multi_rocket,
    xgb_mur
)


models['multi_rocket_xgb'] = clf_xgb_mur



## Train - Test - Evaluate


In [ ]:
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import joblib
import os

for key, value in models.items():
    clf_xgb = value
    model_stats_data = {}
    for key_data, value_data in Datasets.items():              
        # X_train, y_train = prepare_rocket_data(value_data[0], features, target, series_length)
        # X_test, y_test = prepare_rocket_data(value_data[1], features, target, series_length)
        
        X_train = value_data[0].drop(columns=['Pattern'])
        y_train = value_data[0].groupby(level=0)['Pattern'].first().to_frame()
        X_test = value_data[1].drop(columns=['Pattern'])
        y_test = value_data[1].groupby(level=0)['Pattern'].first().to_frame()
                
        
        # Train
        print(f"Training {key_data} {key} ...")
        
        train_start = time.time()
        clf_xgb.fit(X_train, y_train)
        train_end = time.time()
        train_time = train_end - train_start

        model_stats_data[f'{key_data} {key}_train_time'] = train_time

        print(clf_xgb)
        # save the model
        # Create the "Models" directory if it doesn't exist
        if not os.path.exists("Models"):
            os.makedirs("Models")
        joblib.dump(clf_xgb, f"Models/{key_data}_{key}.joblib")
        
        # Test
        print(f"Testing {key_data} {key} ...")
        
        y_train_probs = clf_xgb.predict_proba(X_train) 
        test_start = time.time()
        y_test_probs = clf_xgb.predict_proba(X_test)
        test_end = time.time()
        test_time = test_end - test_start
        model_stats_data[f'{key_data} {key}_test_time'] = test_time
        
        # Evaluate
        
        

        # Training accuracy
        y_train_pred = y_train_probs.argmax(axis=1)  
        train_accuracy = accuracy_score(y_train, y_train_pred)
        # print(f"Training Accuracy: {train_accuracy:.2f}")

        # Testing accuracy
        y_test_pred = y_test_probs.argmax(axis=1)  
        test_accuracy = accuracy_score(y_test, y_test_pred)

        # print(f"Test Accuracy: {test_accuracy:.2f}")
        
        model_stats_data[f'{key_data} {key}_test_accuracy'] = test_accuracy
        model_stats_data[f'{key_data} {key}_train_accuracy'] = train_accuracy
        
        print("Status for {key_data} {key} : /n" , model_stats_data)
        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_test_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=set(y_test), yticklabels=set(y_test))
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.title(f"{key_data} {key}Confusion Matrix")
        plt.show()
    
    model_stats[key] = model_stats_data


In [ ]:
model_stats

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Convert the nested dictionary to a more usable format
def extract_metrics(results_dict):
    data = []
    
    for model_name, configs in results_dict.items():
        for config_name, metrics in configs.items():
            # Extract the configuration type (before the model name)
            config_type = config_name.split(f"{model_name}_")[0].strip()
            
            # Extract specific metrics
            if 'test_accuracy' in config_name:
                metric_type = 'test_accuracy'
                data.append({
                    'model': model_name,
                    'config': config_type,
                    'metric': metric_type,
                    'value': metrics
                })
            elif 'train_accuracy' in config_name:
                metric_type = 'train_accuracy'
                data.append({
                    'model': model_name,
                    'config': config_type,
                    'metric': metric_type,
                    'value': metrics
                })
            elif 'train_time' in config_name:
                metric_type = 'train_time'
                data.append({
                    'model': model_name,
                    'config': config_type,
                    'metric': metric_type,
                    'value': metrics
                })
            elif 'test_time' in config_name:
                metric_type = 'test_time'
                data.append({
                    'model': model_name,
                    'config': config_type,
                    'metric': metric_type,
                    'value': metrics
                })
    
    return pd.DataFrame(data)

# Function to plot grouped bars for a specific metric
def plot_metric(results_dict, metric_name, title, ylabel, figsize=(14, 8)):
    # Extract data
    df = extract_metrics(results_dict)
    df_filtered = df[df['metric'] == metric_name]
    
    # Get unique models and configurations
    models = df_filtered['model'].unique()
    configs = df_filtered['config'].unique()
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Width of a bar 
    bar_width = 0.2
    
    # Set position of bar on X axis
    r = np.arange(len(configs))
    
    # Plot bars for each model
    for i, model in enumerate(models):
        model_data = df_filtered[df_filtered['model'] == model]
        values = []
        for config in configs:
            config_data = model_data[model_data['config'] == config]
            if not config_data.empty:
                values.append(config_data['value'].values[0])
            else:
                values.append(0)
        
        position = [x + i * bar_width for x in r]
        bars = ax.bar(position, values, width=bar_width, label=model)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            if metric_name in ['train_time', 'test_time']:
                # Format time values as seconds
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01 * max(values),
                        f'{height:.1f}s', ha='center', va='bottom', rotation=45, fontsize=8)
            else:
                # Format accuracy values as percentages
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01 * max(values),
                        f'{height:.3f}', ha='center', va='bottom', rotation=45, fontsize=8)
    
    # Add labels and title
    ax.set_xlabel('Configuration Type', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.set_title(f'{title} for Different Models and Configurations', fontweight='bold')
    
    # Set x-axis ticks
    ax.set_xticks([r + (len(models) - 1) * bar_width / 2 for r in range(len(configs))])
    ax.set_xticklabels(configs, rotation=45, ha='right')
    
    # Add legend
    ax.legend(title='Models', loc='lower right')
    
    # Add grid for better readability
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    return fig

# Create dictionary from the provided data
results =model_stats

# Function to plot all metrics
def plot_all_metrics(results_dict):
    # Plot test accuracy
    test_acc_fig = plot_metric(results_dict, 'test_accuracy', 'Test Accuracy', 'Accuracy')
    
    # Plot training accuracy
    train_acc_fig = plot_metric(results_dict, 'train_accuracy', 'Training Accuracy', 'Accuracy')
    
    # Plot training time
    train_time_fig = plot_metric(results_dict, 'train_time', 'Training Time', 'Time (seconds)')
    
    # Plot test/inference time
    test_time_fig = plot_metric(results_dict, 'test_time', 'Inference Time', 'Time (seconds)')
    
    return test_acc_fig, train_acc_fig, train_time_fig, test_time_fig

# Generate and show all the plots
test_acc_fig, train_acc_fig, train_time_fig, test_time_fig = plot_all_metrics(results)

# Display all plots (in a notebook, you would use plt.show() for each figure)
plt.show()

# Pattern Locating


## Import datasets


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils.formatAndPreprocessNewPatterns import normalize_ohlc_len , get_pattern_encoding ,get_reverse_pattern_encoding
from utils.patternLocating import patter_locate_test_data_create

In [ ]:
# file_path = "Datasets/VanilaDataset"
train_patterns = pd.read_csv("Datasets/New_pattern_set_data/train_df.csv")
test_patterns = pd.read_csv("Datasets/New_pattern_set_data/test_df.csv" )

test_patterns_without_no_pattern = test_patterns[test_patterns['Chart Pattern'] != "No Pattern"]
path = 'Datasets/OHLC data'

In [ ]:
pattern_encoding = get_pattern_encoding()
pattern_encoding_reversed = get_reverse_pattern_encoding()
extra_days = 100

test_pattern_segment_wise = patter_locate_test_data_create(train_patterns, test_patterns_without_no_pattern, extra_days,seed=69)

test_pattern_segment_wise.to_csv("Datasets/New_pattern_set_data/test_patterns_large_segment_wise.csv", index=False)


In [ ]:
# test_ids = [0,1,2]
# # count the number of times each test ids are in the Seg_Id column of test_pattern_segment_wise
# test_ids_rows_count_sum = test_pattern_segment_wise[test_pattern_segment_wise['Seg_ID'].isin(test_ids)].groupby('Seg_ID').size().sum()
# test_ids_rows_count_sum

## Loading the models


In [ ]:
# load the pipelined model
import os
import joblib
from utils.FixedLengthTransformer import FixedLengthTransformer
# load the saved models from the Models directory
model_dir = "Models"
model_files = [f for f in os.listdir(model_dir) if f.endswith('.joblib')]
models = {}
for model_file in model_files:
    model_name = model_file.split('.')[0]
    models[model_name] = joblib.load(os.path.join(model_dir, model_file))
    
# print the model names
print("Available models:")
for model_name in models.keys():
    print(model_name)

## Pattern Locating


### Selecting a test samle to test (To reduce time )


In [ ]:
import random
random.seed(69)

test_seg_ids = test_pattern_segment_wise['Seg_ID'].unique()
# select random 100 seg ids from the test_seg_ids
# test_seg_ids = random.sample(list(test_seg_ids), 100)

# selected_test_pattern_segment_wise = test_pattern_segment_wise[test_pattern_segment_wise['Seg_ID'].isin(test_seg_ids)]
selected_test_pattern_segment_wise = test_pattern_segment_wise.copy()

test_ids_rows_count_sum = test_pattern_segment_wise[test_pattern_segment_wise['Seg_ID'].isin(test_seg_ids)].groupby('Seg_ID').size().sum()
test_ids_rows_count_sum , len(test_seg_ids)

In [ ]:

import os

if not os.path.exists("Datasets/New_pattern_set_data"):
    os.makedirs("Datasets/New_pattern_set_data")

# save the test_ids_rows_count_sum and test_seg_ids to a csv file
test_ids_rows_count_sum_df = pd.DataFrame({'test_ids_rows_count_sum': [test_ids_rows_count_sum], 'test_seg_ids': [test_seg_ids]})
test_ids_rows_count_sum_df.to_csv("Datasets/New_pattern_set_data/test_ids_rows_count_sum.csv", index=False)
selected_test_pattern_segment_wise.to_csv("Datasets/New_pattern_set_data/selected_test_pattern_segment_wise.csv", index=False)


### Pattern Locating Parameters


In [ ]:
# window_size = 20
win_size_proportion = 5
padding_proportion = 0.9
stride = 1
probability_threshold = 0.0
# for len norm 
target_len = 30

### Pattern Locating Process


In [ ]:
from utils.functionalPatternLocateAndPlot import functional_pattern_filter_and_point_recognition

In [ ]:
test_seg_ids

In [ ]:
models

In [ ]:
import sys

from tqdm import tqdm

from utils.functionalPatternLocateAndPlot import functional_pattern_filter_and_point_recognition
from utils.patternLocating import cluster_windows, get_ohlc_data_segment, parallel_process_sliding_window, prepare_dataset_for_cluster

selected_models = ['Plane OHLC_mini_rocket_xgb','Length Normed OHLC_mini_rocket_xgb','Width Aug OHLC_mini_rocket_xgb','Length Normed + Width Aug OHLC_mini_rocket_xgb']

located_patterns_and_other_info_dict = {}
window_results_dict = {}

for model_name in selected_models:
    print(f"Selected model: {model_name}")
    model = models[model_name]
    
    located_patterns_and_other_info_list = []
    window_results_list = []

    # for seg_id, group in tqdm(grouped, desc="Processing segments"):
        
    #     test_seg_id = seg_id  
    for test_seg_id in tqdm(test_seg_ids, desc="Processing test segments"):
        try :
            # plot_patterns_for_segment(test_seg_id    , test_pattern_segment_wise)
            grouped = test_pattern_segment_wise.groupby('Seg_ID')
            # Select a group
            group = grouped.get_group(test_seg_id)

            Seg_Start = group.iloc[0]['Seg_Start']
            Seg_End = group.iloc[0]['Seg_End']
            seg_len = (Seg_End - Seg_Start).days

            window_size = seg_len // win_size_proportion
            # print(f"Win size : {window_size}")
            if window_size < 10:
                window_size = 10
            # elif window_size > 30:
            #     window_size = 30
                
            ohlc_data_segment = get_ohlc_data_segment(test_pattern_segment_wise, test_seg_id, path,group) 
            if ohlc_data_segment is None:
                print("OHLC Data segment is empty")
                continue   
            
            len_norm=False
            target_len = target_len
            
            # geet th efirst two words of the model_name 
            model_name_parts = model_name.split(' ')
            data_set_type = model_name_parts[0] + ' ' + model_name_parts[1]
            data_set_type
            
            if data_set_type == "Length Normed":
                len_norm=True
            
            win_results_df = parallel_process_sliding_window(ohlc_data_segment, model, probability_threshold,stride, pattern_encoding_reversed,group,test_seg_id,window_size, padding_proportion, len_norm, target_len)
            
            if win_results_df is None:
                print("Window results dataframe is empty")
                continue
            window_results_list.append(win_results_df)
            # plot_sliding_steps(win_results_df ,ohlc_data_segment,probability_threshold ,test_seg_id)
            predicted_patterns = prepare_dataset_for_cluster(ohlc_data_segment, win_results_df)
            if predicted_patterns is None:
                print("Predicted patterns dataframe is empty")
            # print("Predicted Patterns :",predicted_patterns)
            cluster_labled_windows_df , interseced_clusters_df = cluster_windows(predicted_patterns, probability_threshold, window_size)
            if cluster_labled_windows_df is None or interseced_clusters_df is None:
                print("Clustered windows dataframe is empty")
                continue
            located_patterns_and_other_info = functional_pattern_filter_and_point_recognition(interseced_clusters_df)
            if located_patterns_and_other_info is None:
                print("Located patterns and other info dataframe is empty")
                continue
            # plot_pattern_groups_and_finalized_sections(located_patterns_and_other_info, cluster_labled_windows_df, test_seg_id)
            
            located_patterns_and_other_info_list.append(located_patterns_and_other_info)
        except:
            print("Error in segment ID: ",test_seg_id,"error : ",sys.exc_info()[0])
            continue

    located_patterns_and_other_info_final_df = pd.concat(located_patterns_and_other_info_list)
    window_results_all_df = pd.concat(window_results_list)
    
    located_patterns_and_other_info_dict[model_name] = located_patterns_and_other_info_final_df
    window_results_dict[model_name] = window_results_all_df
    


In [ ]:
#  save the located patterns and other info and window results dataframes to csv files from th edict
if not os.path.exists("Datasets/New_pattern_set_data/1stRun"):
    os.makedirs("Datasets/New_pattern_set_data/1stRun")
for model_name, located_patterns_and_other_info_df in located_patterns_and_other_info_dict.items():
    located_patterns_and_other_info_df.to_csv(f"Datasets/New_pattern_set_data/1stRun/located_patterns_and_other_info_{model_name}.csv", index=False)
    print(f"Saved located patterns and other info dataframe for {model_name} to csv")

for model_name, window_results_df in window_results_dict.items():
    window_results_df.to_csv(f"Datasets/New_pattern_set_data/1stRun/window_results_{model_name}.csv", index=False)
    print(f"Saved window results dataframe for {model_name} to csv")

### Using and Tuning The located Patterns


In [ ]:
import pandas as pd

In [ ]:
# Load all the saved csv files into a dict
located_patterns_and_other_info_dict = {}
window_results_dict = {}
# selected_models = ['Plane OHLC_multi_rocket_xgb','Length Normed OHLC_multi_rocket_xgb','Width Aug OHLC_multi_rocket_xgb']
selected_models = ['Plane OHLC_mini_rocket_xgb','Length Normed OHLC_mini_rocket_xgb','Width Aug OHLC_mini_rocket_xgb','Length Normed + Width Aug OHLC_mini_rocket_xgb']

test_pattern_segment_wise = pd.read_csv("Datasets/New_pattern_set_data/test_patterns_large_segment_wise.csv")
path = 'Datasets/OHLC data'

for model_name in selected_models:
    located_patterns_and_other_info_df= pd.read_csv(f"Datasets/New_pattern_set_data/1stRun/located_patterns_and_other_info_{model_name}.csv")
    window_results_df = pd.read_csv(f"Datasets/New_pattern_set_data/1stRun/window_results_{model_name}.csv")
    
    # convert Start,End and Seg_Start, Seg_End columns to datetime
    window_results_df ['Start'] = pd.to_datetime(window_results_df['Start'])
    window_results_df ['End'] = pd.to_datetime(window_results_df['End'])
    window_results_df['Seg_Start'] = pd.to_datetime(window_results_df['Seg_Start'])
    window_results_df['Seg_End'] = pd.to_datetime(window_results_df['Seg_End'])
    
    # convert Start,End,Seg_Start, Seg_End,Calc_Start, Calc_End columns to datetime
    located_patterns_and_other_info_df ['Start'] = pd.to_datetime(located_patterns_and_other_info_df['Start'])
    located_patterns_and_other_info_df ['End'] = pd.to_datetime(located_patterns_and_other_info_df['End'])
    located_patterns_and_other_info_df['Seg_Start'] = pd.to_datetime(located_patterns_and_other_info_df['Seg_Start'])
    located_patterns_and_other_info_df['Seg_End'] = pd.to_datetime(located_patterns_and_other_info_df['Seg_End'])
    located_patterns_and_other_info_df['Calc_Start'] = pd.to_datetime(located_patterns_and_other_info_df['Calc_Start'])
    located_patterns_and_other_info_df['Calc_End'] = pd.to_datetime(located_patterns_and_other_info_df['Calc_End'])
    
    located_patterns_and_other_info_dict[model_name] = located_patterns_and_other_info_df
    window_results_dict[model_name] = window_results_df

In [ ]:
selected_test_pattern_segment_wise = pd.read_csv("Datasets/New_pattern_set_data/selected_test_pattern_segment_wise.csv")
test_seg_ids = selected_test_pattern_segment_wise['Seg_ID'].unique()
test_seg_ids

In [ ]:
test_pattern_segment_wise

In [ ]:
# convert the Seg_Start and Seg_End , Start and End columns to datetime in test_pattern_segment_wise
test_pattern_segment_wise['Seg_Start'] = pd.to_datetime(test_pattern_segment_wise['Seg_Start'])
test_pattern_segment_wise['Seg_End'] = pd.to_datetime(test_pattern_segment_wise['Seg_End'])
test_pattern_segment_wise['Start'] = pd.to_datetime(test_pattern_segment_wise['Start'])
test_pattern_segment_wise['End'] = pd.to_datetime(test_pattern_segment_wise['End'])



In [ ]:
# window_size = 20
win_size_proportion = 5
padding_proportion = 0.6
stride = 1
probability_threshold = 0.0

In [ ]:
import sys

from tqdm import tqdm

from utils.functionalPatternLocateAndPlot import functional_pattern_filter_and_point_recognition
from utils.patternLocating import cluster_windows, get_ohlc_data_segment, parallel_process_sliding_window, prepare_dataset_for_cluster

located_patterns_and_other_info_updated_dict = {}
for model_name in selected_models:
    print(f"Selected model: {model_name}")
    located_patterns_and_other_info_df = located_patterns_and_other_info_dict[model_name]
    window_results_df = window_results_dict[model_name]
    
    
    located_patterns_and_other_info_updated_list = []
    win_res_grouped_results = window_results_df.groupby('Seg_ID')
    
    for seg_id, win_res_group in win_res_grouped_results:
        print (f"Processing segment ID: {seg_id} out of {len(test_seg_ids)}")
        grouped = test_pattern_segment_wise.groupby('Seg_ID')
        # Select a group
        group = grouped.get_group(seg_id)

        Seg_Start = group.iloc[0]['Seg_Start']
        Seg_End = group.iloc[0]['Seg_End']
        seg_len = (Seg_End - Seg_Start).days

        window_size = seg_len // win_size_proportion
        # print(f"Win size : {window_size}")
        if window_size < 10:
            window_size = 10
        # elif window_size > 30:
        #     window_size = 30
            
        ohlc_data_segment = get_ohlc_data_segment(test_pattern_segment_wise, seg_id, path,group)
        
        # plot_sliding_steps(win_res_group ,ohlc_data_segment,probability_threshold ,seg_id) 
        if ohlc_data_segment is None:
            print("OHLC Data segment is empty")
            continue
        predicted_patterns = prepare_dataset_for_cluster(ohlc_data_segment, win_res_group)
        if predicted_patterns is None:
            print("predicted_patterns is empty")
            continue
        cluster_labled_windows_df , interseced_clusters_df = cluster_windows(predicted_patterns, probability_threshold, window_size)
        if cluster_labled_windows_df is None  or interseced_clusters_df is None:
            print("cluster_labled_windows_df is empty")
            continue
        located_patterns_and_other_info = functional_pattern_filter_and_point_recognition(interseced_clusters_df)
        if (located_patterns_and_other_info is None) | located_patterns_and_other_info.empty:
            print("located_patterns_and_other_info is empty")
            continue
        # drop teh rows of Chart patterns with the name 'No Pattern' , then print how many rows were dropped
        num_no_patterns = len(located_patterns_and_other_info[located_patterns_and_other_info['Chart Pattern'] == 'No Pattern'])
        # if num_no_patterns > 0:
            # print(f"Number of rows dropped for segment ID {seg_id} : {num_no_patterns}")
        located_patterns_and_other_info = located_patterns_and_other_info[located_patterns_and_other_info['Chart Pattern'] != 'No Pattern']

        # plot_pattern_groups_and_finalized_sections(located_patterns_and_other_info, cluster_labled_windows_df, seg_id)
        located_patterns_and_other_info_updated_list.append(located_patterns_and_other_info)
        

    located_patterns_and_other_info_updated_df = pd.concat(located_patterns_and_other_info_updated_list)
    located_patterns_and_other_info_updated_dict[model_name] = located_patterns_and_other_info_updated_df
    


In [ ]:
import os
# save located_patterns_and_other_info_updated_df s to csv files
if not os.path.exists("Datasets/New_pattern_set_data/2ndRun"):
    os.makedirs("Datasets/New_pattern_set_data/2ndRun")
for model_name, located_patterns_and_other_info_updated_df in located_patterns_and_other_info_updated_dict.items():
    located_patterns_and_other_info_updated_df.to_csv(f"Datasets/New_pattern_set_data/2ndRun/located_patterns_and_other_info_updated_{model_name}.csv", index=False)
    print(f"Saved located patterns and other info updated dataframe for {model_name} to csv")

### Plot the process for a sample


In [ ]:
from utils.patternLocating import plot_patterns_for_segment

test_seg_id = test_seg_ids[-1]
plot_patterns_for_segment(test_seg_id , test_pattern_segment_wise)

In [ ]:
# ohlc_data_segment columns data types
window_results_df.dtypes

In [ ]:
len(window_results_df['Seg_ID'].unique())

In [ ]:
win_res_grouped_results.get_group(146)

In [ ]:
import random

from utils.functionalPatternLocateAndPlot import plot_pattern_groups_and_finalized_sections
from utils.patternLocating import plot_sliding_steps


for model_name in selected_models:
    print(f"Selected model: {model_name}")
    
    located_patterns_and_other_info_df = located_patterns_and_other_info_dict[model_name]
    window_results_df = window_results_dict[model_name]
    
    win_res_grouped_results = window_results_df.groupby('Seg_ID')
    # for seg_id, win_res_group in win_res_grouped_results:
    grouped = test_pattern_segment_wise.groupby('Seg_ID')
    # Select a group

    # seg_id = random.choice(window_results_all_df['Seg_ID'].unique())
    # seg_id = seg_ids[0]
    seg_id = test_seg_id
    win_res_group = win_res_grouped_results.get_group(seg_id)

    group = grouped.get_group(seg_id)
        
    ohlc_data_segment = get_ohlc_data_segment(test_pattern_segment_wise, seg_id, path,group)

    plot_sliding_steps(win_res_group ,ohlc_data_segment,0 ,seg_id,save = True) 
    if ohlc_data_segment is None:
        print("OHLC Data segment is empty")
        # continue
    predicted_patterns = prepare_dataset_for_cluster(ohlc_data_segment, win_res_group)
    if predicted_patterns is None:
        print("predicted_patterns is empty")
        # continue
    cluster_labled_windows_df , interseced_clusters_df = cluster_windows(predicted_patterns, probability_threshold, window_size)
    if cluster_labled_windows_df is None  or interseced_clusters_df is None:
        print("cluster_labled_windows_df is empty")
        # continue
    located_patterns_and_other_info = functional_pattern_filter_and_point_recognition(interseced_clusters_df)
    if located_patterns_and_other_info is None:
        print("located_patterns_and_other_info is empty")
        # continue
    plot_pattern_groups_and_finalized_sections(located_patterns_and_other_info, cluster_labled_windows_df, seg_id)
    # located_patterns_and_other_info_updated_list.append(located_patterns_and_other_info)


## Evaluate the result


In [ ]:
from utils.eval import intersection_over_union , mean_abselute_error
from utils.formatAndPreprocessNewPatterns import get_pattern_list
chart_patterns = get_pattern_list()

In [ ]:
test_patterns = pd.read_csv("Datasets/New_pattern_set_data/test_df.csv" )

test_patterns_without_no_pattern = test_patterns[test_patterns['Chart Pattern'] != "No Pattern"].copy()
# rename the Unnamed: 0 column to Instance
test_patterns_without_no_pattern.rename(columns={'Unnamed: 0': 'Instance'}, inplace=True)
# rename the Unnamed: 0 column to Instance
selected_test_pattern_segment_wise.rename(columns={'Unnamed: 0': 'Instance'}, inplace=True)
selected_instances = selected_test_pattern_segment_wise['Instance'].unique()
selected_test_patterns_without_no_pattern = test_patterns_without_no_pattern[test_patterns_without_no_pattern['Instance'].isin(selected_instances)]

In [ ]:
selected_test_pattern_segment_wise = pd.read_csv("Datasets/New_pattern_set_data/selected_test_pattern_segment_wise.csv")

test_ids_rows_count_sum = selected_test_pattern_segment_wise.groupby('Seg_ID').size().sum()

# per pattern row count in to a dict
pattern_row_count = {}
for pattern in chart_patterns:
    pattern_row_count[pattern] = selected_test_pattern_segment_wise[selected_test_pattern_segment_wise['Chart Pattern'] == pattern].shape[0]
    
pattern_row_count

In [ ]:
selected_test_pattern_segment_wise

In [ ]:
from tqdm import tqdm
import pandas as pd
def get_model_eval_res(located_patterns_and_other_info_updated_dict,window_results_dict,selected_models):
    model_eval_results_dict = {}
    for model_name in selected_models:
        print(f"\n Selected model: {model_name}")
        
        located_patterns_and_other_info_updated_df = located_patterns_and_other_info_updated_dict[model_name]
        window_results_df = window_results_dict[model_name]

        # dictionary to store the count of properly located patterns , iou and mae for each properly detected pattern for each model
        

        # Dictionary to store the count of properly located patterns
        number_of_properly_located_patterns = {}
        iou_for_each_properly_detected_pattern = {}
        mae_for_each_properly_detected_pattern = {}

        # Convert date columns to datetime (once, outside the loop for efficiency)
        located_patterns_and_other_info_updated_df['Calc_Start'] = pd.to_datetime(located_patterns_and_other_info_updated_df['Calc_Start'])
        located_patterns_and_other_info_updated_df['Calc_End'] = pd.to_datetime(located_patterns_and_other_info_updated_df['Calc_End'])

        # Iterate over test patterns with progress bar
        for index, row in selected_test_patterns_without_no_pattern.iterrows():
            sys.stdout.write(f"\rProcessing row {index + 1}/{len(selected_test_patterns_without_no_pattern)}")
            sys.stdout.flush() 
            symbol = row['Symbol']
            chart_pattern = row['Chart Pattern']
            start_date = pd.to_datetime(row['Start']).tz_localize(None)
            end_date = pd.to_datetime(row['End']).tz_localize(None)
            
            # Filter for matching symbol and chart pattern
            located_patterns_for_this = located_patterns_and_other_info_updated_df[
                (located_patterns_and_other_info_updated_df['Symbol'] == symbol) &
                (located_patterns_and_other_info_updated_df['Chart Pattern'] == chart_pattern)
            ].copy()  # Use `.copy()` to avoid SettingWithCopyWarning
            
            if located_patterns_for_this.empty:
                continue  # Skip if no matching rows
            
            # Compute IoU for each row using .loc to avoid warnings
            located_patterns_for_this.loc[:, 'IoU'] = located_patterns_for_this.apply(
                lambda x: intersection_over_union(start_date, end_date, x['Calc_Start'], x['Calc_End']),
                axis=1
            )
            
            # Compute MAE for each row using .loc to avoid warnings
            located_patterns_for_this.loc[:, 'MAE'] = located_patterns_for_this.apply(
                lambda x: mean_abselute_error(start_date, end_date, x['Calc_Start'], x['Calc_End']),
                axis=1
            )

            
            # Filter based on IoU threshold (≥ 0.8)
            located_patterns_for_this_proper = located_patterns_for_this[located_patterns_for_this['IoU'] >= 0.25]
            
            if not located_patterns_for_this_proper.empty:
                number_of_properly_located_patterns[chart_pattern] = number_of_properly_located_patterns.get(chart_pattern, 0) + 1
                iou_for_each_properly_detected_pattern[chart_pattern] = iou_for_each_properly_detected_pattern.get(chart_pattern, 0) + max(located_patterns_for_this_proper['IoU'])
                mae_for_each_properly_detected_pattern[chart_pattern] = mae_for_each_properly_detected_pattern.get(chart_pattern, 0) + min(located_patterns_for_this_proper['MAE'])

        number_of_properly_located_patterns
        
        model_eval_results_dict[model_name] = {
            'number_of_properly_located_patterns': number_of_properly_located_patterns,
            'iou_for_each_properly_detected_pattern': iou_for_each_properly_detected_pattern,
            'mae_for_each_properly_detected_pattern': mae_for_each_properly_detected_pattern
        }
    return model_eval_results_dict

model_eval_results_dict = get_model_eval_res(located_patterns_and_other_info_updated_dict,window_results_dict,selected_models)

In [ ]:
model_eval_results_dict

### Total Evaluations for the current params


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

def create_comprehensive_model_comparison(all_models_metrics):
    """
    Create a comprehensive visualization comparing all models across all metrics,
    using nested concentric pie charts for Precision and Recall.
    
    Parameters:
    -----------
    all_models_metrics : dict
        Dictionary containing metrics for each model
    """
    models = list(all_models_metrics.keys())
    n_models = len(models)
    
    # Define the metrics to include
    key_metrics = {
        'total_recall': 'Recall',
        'total_precision': 'Precision', 
        'overall_f1': 'F1 Score',
        'overall_iou': 'IoU',
        'overall_mae': 'MAE'
    }
    
    # Create figure with GridSpec for flexible layout
    fig = plt.figure(figsize=(20, 14))
    
    # Add main title with enough space for legend below it
    plt.suptitle('Comprehensive Model Evaluation', fontsize=16, y=0.98)
    
    # Define a color palette for models
    colors = plt.cm.tab10(np.linspace(0, 1, n_models))
    
    # Create a master legend below the title
    legend_handles = [plt.Line2D([0], [0], color=colors[i], lw=4, label=model) for i, model in enumerate(models)]
    fig.legend(
        handles=legend_handles,
        labels=models,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.93),  # Moved down from 0.98 to 0.93
        ncol=n_models,
        fontsize=12
    )
    
    # Adjust GridSpec to account for the title and legend
    gs = GridSpec(3, 3, figure=fig, height_ratios=[1.2, 1.2, 1], top=0.88)  # Reduced top from 0.95 to 0.88
    

    
    # 1. Precision Nested Pie Chart - top left
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Create a multi-layer nested pie chart for precision
    # Each ring represents a different model
    precision_values = [metrics['total_precision'] for metrics in all_models_metrics.values()]
    
    # Calculate radii for each ring (outermost ring is largest)
    radii = np.linspace(0.5, 1.0, n_models+1)[1:]  # start from second element to skip 0.5
    
    # Plot each model as a ring, outermost = first model
    for i, model in enumerate(models):
        # Create data for this model's ring [precision, 1-precision]
        data = [precision_values[i], 1-precision_values[i]]
        colors_ring = [colors[i], 'lightgray']
        
        # Create pie chart for this ring
        wedges, texts = ax1.pie(
            data, 
            radius=radii[i],
            colors=colors_ring,
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=0.15, edgecolor='w')
        )
        
        # Add only the value (no model name) to the pie chart wedge
        angle = (wedges[0].theta1 + wedges[0].theta2) / 2
        x = (radii[i] - 0.075) * np.cos(np.radians(angle))
        y = (radii[i] - 0.075) * np.sin(np.radians(angle))
        ax1.text(x, y, f"{precision_values[i]:.3f}", 
                ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Create center circle for donut effect
    centre_circle = plt.Circle((0, 0), 0.25, fc='white')
    ax1.add_patch(centre_circle)
    
    ax1.set_title('Precision Comparison (Higher is Better)')
    ax1.set_aspect('equal')
    
    # 2. Recall Nested Pie Chart - top middle
    ax2 = fig.add_subplot(gs[0, 1])
    
    # Create a multi-layer nested pie chart for recall
    recall_values = [metrics['total_recall'] for metrics in all_models_metrics.values()]
    
    # Plot each model as a ring, outermost = first model
    for i, model in enumerate(models):
        # Create data for this model's ring [recall, 1-recall]
        data = [recall_values[i], 1-recall_values[i]]
        colors_ring = [colors[i], 'lightgray']
        
        # Create pie chart for this ring
        wedges, texts = ax2.pie(
            data, 
            radius=radii[i],
            colors=colors_ring,
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=0.15, edgecolor='w')
        )
        
        # Add only the value (no model name) to the pie chart wedge
        angle = (wedges[0].theta1 + wedges[0].theta2) / 2
        x = (radii[i] - 0.075) * np.cos(np.radians(angle))
        y = (radii[i] - 0.075) * np.sin(np.radians(angle))
        ax2.text(x, y, f"{recall_values[i]:.3f}", 
                ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Create center circle for donut effect
    centre_circle = plt.Circle((0, 0), 0.25, fc='white')
    ax2.add_patch(centre_circle)
    
    ax2.set_title('Recall Comparison (Higher is Better)')
    ax2.set_aspect('equal')
    
    # 3. F1 Score and IoU - top right
    ax3 = fig.add_subplot(gs[0, 2])
    
    # Prepare data for grouped bar chart
    metrics_to_plot = ['overall_f1', 'overall_iou']
    x = np.arange(len(metrics_to_plot))
    width = 0.8 / n_models
    
    # Plot grouped bars for each model
    for i, (model_name, metrics) in enumerate(all_models_metrics.items()):
        values = [metrics[key] for key in metrics_to_plot]
        bars = ax3.bar(x + i*width - width*(n_models-1)/2, values, width, color=colors[i])
        
        # Add value labels above each bar
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', fontsize=9, rotation=0)
    
    # Customize the plot
    ax3.set_xticks(x)
    ax3.set_xticklabels([key_metrics[key] for key in metrics_to_plot])
    ax3.set_ylabel('Score')
    ax3.set_title('F1 Score & IoU Comparison (Higher is Better)')
    ax3.set_ylim(0, 1.0)
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 4. MAE comparison (separate bar chart) - middle left
    ax4 = fig.add_subplot(gs[1, 0])
    
    mae_values = [metrics['overall_mae'] for metrics in all_models_metrics.values()]
    bars = ax4.bar(models, mae_values, color=colors)
    
    # Add value labels above MAE bars
    for bar, value in zip(bars, mae_values):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    
    ax4.set_ylabel('Error')
    ax4.set_title('Mean Absolute Error (Lower is Better)')
    ax4.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 5. Model metrics radar chart - middle center
    ax5 = fig.add_subplot(gs[1, 1], polar=True)
    
    # Setup for radar chart
    metrics_for_radar = ['total_recall', 'total_precision', 'overall_f1', 'overall_iou']
    num_vars = len(metrics_for_radar)
    angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]  # Close the loop
    
    # Plot each model on the radar chart
    for i, (model_name, metrics) in enumerate(all_models_metrics.items()):
        values = [metrics[metric] for metric in metrics_for_radar]
        values += values[:1]  # Close the loop
        
        ax5.plot(angles, values, linewidth=2, linestyle='solid', color=colors[i])
        ax5.fill(angles, values, alpha=0.1, color=colors[i])
    
    # Set radar chart labels
    ax5.set_xticks(angles[:-1])
    ax5.set_xticklabels([key_metrics[metric] for metric in metrics_for_radar])
    ax5.set_ylim(0, 1)
    ax5.set_title('Model Performance Radar Chart')
    
    # 6. Model comparison bar - middle right
    ax6 = fig.add_subplot(gs[1, 2])
    
    # Calculate the average of the four main metrics for an overall score
    # (excluding MAE which is inverse, lower is better)
    overall_scores = []
    for model_name, metrics in all_models_metrics.items():
        score = (metrics['total_recall'] + metrics['total_precision'] + 
                metrics['overall_f1'] + metrics['overall_iou']) / 4
        overall_scores.append(score)
    
    # Create horizontal bar chart
    y_pos = np.arange(len(models))
    ax6.barh(y_pos, overall_scores, color=colors)
    ax6.set_yticks(y_pos)
    ax6.set_yticklabels(models)
    ax6.invert_yaxis()  # labels read top-to-bottom
    ax6.set_xlabel('Overall Performance Score')
    ax6.set_title('Overall Model Comparison (Higher is Better)')
    
    # Add value labels
    for i, v in enumerate(overall_scores):
        ax6.text(v + 0.01, i, f'{v:.3f}', va='center')
    
    # 7. Detailed per-model metrics table - bottom span all columns
    ax7 = fig.add_subplot(gs[2, :])
    ax7.axis('tight')
    ax7.axis('off')
    
    # Prepare table data
    table_data = []
    for model_name, metrics in all_models_metrics.items():
        row = [model_name]
        for key in key_metrics:
            row.append(f"{metrics[key]:.4f}")
        table_data.append(row)
    
    # Create table
    column_labels = ['Model'] + list(key_metrics.values())
    table = ax7.table(
        cellText=table_data,
        colLabels=column_labels,
        loc='center',
        cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    ax7.set_title('Model Metrics Summary Table')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.88])  # Adjusted rect to account for title and legend
    
    plt.show()
    
    return fig

# The evaluate_model and evaluate_all_models functions remain unchanged
# The evaluate_model and evaluate_all_models functions remain unchanged
# The evaluate_model function remains unchanged from your second code snippet
def evaluate_model(model_name, model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_and_other_info_updated_dict):
    """Evaluate a model and calculate metrics without redundant plots"""
    print(f"\n{'='*20} Model: {model_name} {'='*20}")
    
    # Extract model results
    number_of_properly_located_patterns = model_eval_results_dict[model_name]['number_of_properly_located_patterns']
    located_patterns_df = located_patterns_and_other_info_updated_dict[model_name]
    mae_for_each_properly_detected_pattern = model_eval_results_dict[model_name]['mae_for_each_properly_detected_pattern']
    iou_for_each_properly_detected_pattern = model_eval_results_dict[model_name]['iou_for_each_properly_detected_pattern']
    
    # Calculate metrics without plotting
    # Recall
    total_number_of_all_patterns = sum(pattern_row_count.values())
    total_number_of_properly_located_patterns = sum(number_of_properly_located_patterns.values())
    total_recall = total_number_of_properly_located_patterns / total_number_of_all_patterns if total_number_of_all_patterns > 0 else 0
    
    per_pattern_recall = {}
    for pattern, count in number_of_properly_located_patterns.items():
        pattern_count = test_patterns[test_patterns['Chart Pattern'] == pattern].shape[0]
        if pattern_count > 0:
            per_pattern_recall[pattern] = count / pattern_count
        else:
            per_pattern_recall[pattern] = 0
    
    # Precision
    total_number_of_all_located_patterns = len(located_patterns_df)
    total_precision = total_number_of_properly_located_patterns / total_number_of_all_located_patterns if total_number_of_all_located_patterns > 0 else 0
    
    per_pattern_precision = {}
    for pattern, count in number_of_properly_located_patterns.items():
        pattern_predictions = located_patterns_df[located_patterns_df['Chart Pattern'] == pattern].shape[0]
        if pattern_predictions > 0:
            per_pattern_precision[pattern] = count / pattern_predictions
        else:
            per_pattern_precision[pattern] = 0
    
    # F1 Score
    per_pattern_f1 = {}
    for pattern in per_pattern_recall.keys():
        precision = per_pattern_precision.get(pattern, 0)
        recall = per_pattern_recall.get(pattern, 0)
        if precision + recall > 0:
            per_pattern_f1[pattern] = 2 * (precision * recall) / (precision + recall)
        else:
            per_pattern_f1[pattern] = 0
    
    all_precisions = list(per_pattern_precision.values())
    all_recalls = list(per_pattern_recall.values())
    avg_precision = sum(all_precisions) / len(all_precisions) if all_precisions else 0
    avg_recall = sum(all_recalls) / len(all_recalls) if all_recalls else 0
    
    if avg_precision + avg_recall == 0:
        overall_f1 = 0
    else:
        overall_f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)
    
    # MAE
    per_pattern_mae = {}
    for pattern, count in number_of_properly_located_patterns.items():
        if count > 0:
            per_pattern_mae[pattern] = mae_for_each_properly_detected_pattern.get(pattern, 0) / count
        else:
            per_pattern_mae[pattern] = 0
    
    total_mae_sum = sum(mae_for_each_properly_detected_pattern.values())
    total_proper_patterns = sum(number_of_properly_located_patterns.values())
    overall_mae = total_mae_sum / total_proper_patterns if total_proper_patterns > 0 else 0
    
    # IoU
    per_pattern_iou = {}
    for pattern, count in number_of_properly_located_patterns.items():
        if count > 0:
            per_pattern_iou[pattern] = iou_for_each_properly_detected_pattern.get(pattern, 0) / count
        else:
            per_pattern_iou[pattern] = 0
    
    total_iou_sum = sum(iou_for_each_properly_detected_pattern.values())
    overall_iou = total_iou_sum / total_proper_patterns if total_proper_patterns > 0 else 0
    
    # Print summary of metrics
    print(f"Overall Recall: {total_recall:.4f}")
    print(f"Overall Precision: {total_precision:.4f}")
    print(f"Overall F1 Score: {overall_f1:.4f}")
    print(f"Overall Mean Absolute Error: {overall_mae:.4f}")
    print(f"Overall Mean Intersection over Union: {overall_iou:.4f}")
    
    # Store all metrics in one place for easy access
    metrics_summary = {
        'total_recall': total_recall,
        'per_pattern_recall': per_pattern_recall,
        'total_precision': total_precision,
        'per_pattern_precision': per_pattern_precision,
        'overall_f1': overall_f1,
        'per_pattern_f1': per_pattern_f1,
        'overall_mae': overall_mae,
        'per_pattern_mae': per_pattern_mae,
        'overall_iou': overall_iou,
        'per_pattern_iou': per_pattern_iou
    }
    
    return metrics_summary

# Updated evaluate_all_models function that only creates the comprehensive plot
def evaluate_all_models(model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_and_other_info_updated_dict):
    """Evaluate all models and return metrics summary with comprehensive plot only"""
    all_models_metrics = {}
    
    for model_name in model_eval_results_dict.keys():
        all_models_metrics[model_name] = evaluate_model(
            model_name,
            model_eval_results_dict,
            pattern_row_count,
            test_patterns,
            located_patterns_and_other_info_updated_dict
        )
    
    # Only create the comprehensive visualization
    if len(model_eval_results_dict) > 0:
        print("\n--- Comprehensive Model Comparison ---")
        figure = create_comprehensive_model_comparison(all_models_metrics)
    
    return all_models_metrics , figure

# Example of how to use the functions:
all_metrics = evaluate_all_models(model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_and_other_info_updated_dict)
figure = all_metrics[1]
# save the figure in Results_Plots folder
if not os.path.exists("Results_Plots"):
    os.makedirs("Results_Plots")

def get_short_model_code(model_name):
    prefix, suffix = model_name.split('_', 1)
    initials = ''.join([word[0] for word in prefix.replace('+', '').split()])
    suffix_word = suffix.split('_')[1]  # pick second word, e.g., 'rocket' in 'mini_rocket_xgb'
    return initials + suffix_word

short_model_names = [get_short_model_code(name) for name in selected_models]
short_model_str = '_'.join(short_model_names)

file_name = f"Results_Plots/Comp_Comparison_ws-{win_size_proportion}_pp-{padding_proportion}_s-{stride}_pt-{probability_threshold}_models-{short_model_str}"

# give it a rename of win_size_proportion,padding_proportion ,stride ,probability_threshold parameter values and the set of models that used 
# for the evaluation
figure.savefig(f"{file_name}.png", bbox_inches='tight')
# convert all_metrics[0] to a dataframe 
all_metrics_df = pd.DataFrame(all_metrics[0]).T
all_metrics_df.reset_index(drop=False, inplace=True)
# rename "Index" column to model
all_metrics_df.rename(columns={'index': 'Model'}, inplace=True)
all_metrics_df['win_size_proportion'] = win_size_proportion
all_metrics_df['padding_proportion'] = padding_proportion
all_metrics_df['stride'] = stride
all_metrics_df['probability_threshold'] = probability_threshold
# add overall score to each model
for model_name, metrics in all_metrics[0].items():
    score = (metrics['total_recall'] + metrics['total_precision'] + 
            metrics['overall_f1'] + metrics['overall_iou']) / 4
    all_metrics_df.loc[all_metrics_df['Model'] == model_name, 'Overall Score'] = score
all_metrics_df.to_csv(f"{file_name}.csv", index=True)

### Param Tuning for locating patterns


In [ ]:
import math
import sys
import os
import pandas as pd
import numpy as np
import itertools
from IPython.display import clear_output
from tqdm import tqdm

from utils.functionalPatternLocateAndPlot import functional_pattern_filter_and_point_recognition
from utils.patternLocating import cluster_windows, get_ohlc_data_segment, parallel_process_sliding_window, prepare_dataset_for_cluster

# save the figure in Patter_Loc_Results folder
if not os.path.exists("Patter_Loc_Results"):
    os.makedirs("Patter_Loc_Results")
    os.makedirs("Patter_Loc_Results/Plots")
    os.makedirs("Patter_Loc_Results/Dataframe")

# Load all the saved csv files into a dict
located_patterns_and_other_info_dict = {}
window_results_dict = {}
selected_models = ['Plane OHLC_mini_rocket_xgb','Length Normed OHLC_mini_rocket_xgb','Width Aug OHLC_mini_rocket_xgb','Length Normed + Width Aug OHLC_mini_rocket_xgb']


for model_name in selected_models:
    located_patterns_and_other_info_dict[model_name] = pd.read_csv(f"Datasets/New_pattern_set_data/1stRun/located_patterns_and_other_info_{model_name}.csv")
    window_results_dict[model_name] = pd.read_csv(f"Datasets/New_pattern_set_data/1stRun/window_results_{model_name}.csv")
    
selected_test_pattern_segment_wise = pd.read_csv("Datasets/New_pattern_set_data/selected_test_pattern_segment_wise.csv")

test_ids_rows_count_sum = selected_test_pattern_segment_wise.groupby('Seg_ID').size().sum()

# per pattern row count in to a dict
pattern_row_count = {}
for pattern in chart_patterns:  # chart_patterns seems to be defined elsewhere
    pattern_row_count[pattern] = selected_test_pattern_segment_wise[selected_test_pattern_segment_wise['Chart Pattern'] == pattern].shape[0]

win_size_proportion = 5
max_pad = 0.9

# Define the parameter ranges with single decimal precision
probability_thresholds = [round(x, 1) for x in np.arange(0.0, 1.0, 0.1)]
padding_proportions = [round(x, 1) for x in np.arange(0.0, max_pad, 0.1)]
strides = list(range(1, 10, 1))

all_metrics_list = []

# Use itertools for cleaner parameter iteration
parameter_combinations = list(itertools.product(probability_thresholds, padding_proportions, strides))
total_combinations = len(parameter_combinations)

# Main loop with output clearing
for i, (probability_threshold, padding_proportion, stride) in enumerate(parameter_combinations):
    # Clear previous outputs to avoid cluttering the notebook
    clear_output(wait=True)
    
    # Show progress information
    print(f"Progress: {i+1}/{total_combinations} combinations ({((i+1)/total_combinations)*100:.1f}%)")
    print(f"Parameters: probability_threshold={probability_threshold:.1f}, padding_proportion={padding_proportion:.1f}, stride={stride}")
    
    located_patterns_and_other_info_updated_dict = {}
    for model_name in selected_models:
        print(f"Processing model: {model_name}")
        located_patterns_and_other_info_df = located_patterns_and_other_info_dict[model_name]
        window_results_df = window_results_dict[model_name]
        
        located_patterns_and_other_info_updated_list = []
        win_res_grouped_results = window_results_df.groupby('Seg_ID')
        
        # Track segment processing with a nested progress bar
        seg_count = len(win_res_grouped_results)
        seg_counter = 0
        
        for seg_id, win_res_group in win_res_grouped_results:
            seg_counter += 1
            if seg_counter % 10 == 0:  # Update less frequently to reduce output
                print(f"  - Processing segment {seg_counter}/{seg_count}", end='\r')
                
            grouped = test_pattern_segment_wise.groupby('Seg_ID')  # test_pattern_segment_wise needs to be defined
            # Select a group
            try:
                group = grouped.get_group(seg_id)
            except KeyError:
                continue  # Skip if segment ID not found
                
            Seg_Start = group.iloc[0]['Seg_Start']
            Seg_End = group.iloc[0]['Seg_End']
            seg_len = (Seg_End - Seg_Start).days

            window_size = seg_len // win_size_proportion
            if window_size < 10:
                window_size = 10
                
            pad_step_count = math.ceil(window_size * padding_proportion)
            already_padded = math.ceil(window_size * max_pad)
            
            missing_pad = already_padded - pad_step_count
            
            padded_win_res_group = win_res_group.copy()
            padded_win_res_group = padded_win_res_group.iloc[missing_pad:-missing_pad]
            
            # Apply stride
            padded_strided_win_res_group = padded_win_res_group.iloc[::stride, :]
            
            # reset the index
            padded_strided_win_res_group.reset_index(drop=True, inplace=True)
            
            try:
                ohlc_data_segment = get_ohlc_data_segment(test_pattern_segment_wise, seg_id, path, group)  # path needs to be defined
                
                if ohlc_data_segment is None:
                    continue
                    
                predicted_patterns = prepare_dataset_for_cluster(ohlc_data_segment, padded_strided_win_res_group)
                if predicted_patterns is None:
                    continue
                    
                cluster_labled_windows_df, interseced_clusters_df = cluster_windows(predicted_patterns, probability_threshold, window_size)
                if cluster_labled_windows_df is None or interseced_clusters_df is None:
                    continue
                    
                located_patterns_and_other_info = functional_pattern_filter_and_point_recognition(interseced_clusters_df)
                if (located_patterns_and_other_info is None) or located_patterns_and_other_info.empty:
                    continue
                    
                # Filter out 'No Pattern' entries
                located_patterns_and_other_info = located_patterns_and_other_info[located_patterns_and_other_info['Chart Pattern'] != 'No Pattern']
                if not located_patterns_and_other_info.empty:
                    located_patterns_and_other_info_updated_list.append(located_patterns_and_other_info)
            except Exception as e:
                print(f"Error processing segment {seg_id}: {str(e)}")
                continue

        print(f"Completed processing for model: {model_name}                ")  # Extra spaces to clear the line
        
        if located_patterns_and_other_info_updated_list:
            located_patterns_and_other_info_updated_df = pd.concat(located_patterns_and_other_info_updated_list)
            located_patterns_and_other_info_updated_dict[model_name] = located_patterns_and_other_info_updated_df
        else:
            print(f"Warning: No valid patterns found for model {model_name} with these parameters")
            located_patterns_and_other_info_updated_dict[model_name] = pd.DataFrame()  # Empty dataframe
    
    try:
        model_eval_results_dict = get_model_eval_res(located_patterns_and_other_info_updated_dict, window_results_dict, selected_models)
        all_metrics = evaluate_all_models(model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_and_other_info_updated_dict)
        figure = all_metrics[1]

        # Format file names with single decimal point
        plot_path = f"Patter_Loc_Results/Plots/Comprehensive_Model_Comparison_win_size_proportion-{win_size_proportion}_padding_proportion-{padding_proportion:.1f}_stride-{stride}_probability_threshold-{probability_threshold:.1f}_models-{selected_models}.png"
        # figure.savefig(plot_path, bbox_inches='tight')
        
        # convert all_metrics[0] to a dataframe 
        all_metrics_df = pd.DataFrame(all_metrics[0]).T
        all_metrics_df.reset_index(drop=False, inplace=True)
        # rename "Index" column to model
        all_metrics_df.rename(columns={'index': 'Model'}, inplace=True)
        all_metrics_df['win_size_proportion'] = win_size_proportion
        all_metrics_df['padding_proportion'] = round(padding_proportion, 1)
        all_metrics_df['stride'] = stride
        all_metrics_df['probability_threshold'] = round(probability_threshold, 1)
        all_metrics_df['plot_path'] = plot_path
        
        # Format CSV filenames with single decimal point
        csv_path = f"Patter_Loc_Results/Dataframe/Comprehensive_Model_Comparison_win_size_proportion-{win_size_proportion}_padding_proportion-{padding_proportion:.1f}_stride-{stride}_probability_threshold-{probability_threshold:.1f}_models-{selected_models}.csv"
        # all_metrics_df.to_csv(csv_path, index=True)
        all_metrics_list.append(all_metrics_df)
        
        print(f"Saved results for parameters: prob={probability_threshold:.1f}, pad={padding_proportion:.1f}, stride={stride}")
    except Exception as e:
        print(f"Error processing combination {probability_threshold:.1f}, {padding_proportion:.1f}, {stride}: {str(e)}")
        continue

# Final summary after all processing is complete
clear_output(wait=True)
print("All parameter combinations processed!")
print(f"Total successful combinations: {len(all_metrics_list)}/{total_combinations}")

# Combine all results
if all_metrics_list:
    all_metrics_df_conc = pd.concat(all_metrics_list)
    all_metrics_df_conc.to_csv(f"Patter_Loc_Results/Dataframe/Comprehensive_Model_Comparison_All.csv", index=True)
    display(all_metrics_df_conc.head())  # Show a preview of the results
else:
    print("No successful parameter combinations found.")

In [ ]:
# create a new data frame by droping all the per pattern columns from all_metrics_df_conc
all_metrics_df_conc_summary = all_metrics_df_conc.drop(columns=['per_pattern_recall','per_pattern_precision','per_pattern_f1','per_pattern_mae','per_pattern_iou','plot_path'])
all_metrics_df_conc_summary.to_csv(f"Patter_Loc_Results/Dataframe/Comprehensive_Model_Comparison_All_Summary.csv", index=True)
all_metrics_df_conc_summary


In [ ]:
df = all_metrics_df_conc_summary.copy()

# Function to calculate normalized score across metrics
def calculate_overall_score(row):
    # For metrics where higher is better
    recall_norm = row['total_recall'] / df['total_recall'].max()
    precision_norm = row['total_precision'] / df['total_precision'].max()
    f1_norm = row['overall_f1'] / df['overall_f1'].max() 
    iou_norm = row['overall_iou'] / df['overall_iou'].max()
    
    # For metrics where lower is better
    mae_norm = 1 - (row['overall_mae'] / df['overall_mae'].max())
    
    # Weighted average - adjust weights based on importance
    weights = {'recall': 0.25, 'precision': 0.25, 'f1': 0.2, 'mae': 0.1, 'iou': 0.2}
    
    return (recall_norm * weights['recall'] + 
            precision_norm * weights['precision'] +
            f1_norm * weights['f1'] + 
            mae_norm * weights['mae'] + 
            iou_norm * weights['iou'])

# Add overall score column
all_metrics_df_conc_summary['overall_score'] = all_metrics_df_conc_summary.apply(calculate_overall_score, axis=1)

In [ ]:
all_metrics_df_conc_summary.reset_index(drop=False, inplace=True)
all_metrics_df_conc_summary

In [ ]:
# # sort all_metrics_df_conc_summary by overall_score in descending order
# all_metrics_df_conc_summary.sort_values(by='overall_score', ascending=False, inplace=True)
# all_metrics_df_conc_summary

In [ ]:
import os

def create_metric_dashboard(df, metric_name, higher_better=True):
    """Create a unified dashboard for a specific performance metric."""
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec
    import seaborn as sns
    import pandas as pd
    import numpy as np
    from IPython.display import Image, display

    # Generate a full model-based folder name
    model_names = sorted(df['Model'].unique())  # Sort for consistency
    full_model_name = "-".join(model_names)  # Join full names with '-'
    
    # Handle overly long folder names by shortening model names
    if len(full_model_name) > 100:  
        short_model_name = "-".join([name[:6] for name in model_names])  # Take first 6 letters
    else:
        short_model_name = full_model_name

    save_folder = f"Patter_Loc_Results/Plots/{short_model_name}"
    
    # Ensure the directory exists
    os.makedirs(save_folder, exist_ok=True)

    # Filter parameters
    params = ['padding_proportion', 'stride', 'probability_threshold']
    
    # Create figure with custom layout
    fig = plt.figure(figsize=(24, 18))
    gs = GridSpec(3, 4, figure=fig)
    
    # Parameter Impact Analysis (Top Row)
    axs = [fig.add_subplot(gs[0, i]) for i in range(3)]
    for i, param in enumerate(params):
        sns.scatterplot(data=df, x=param, y=metric_name, hue='Model', 
                        alpha=0.7, ax=axs[i], palette='viridis')
        sns.lineplot(data=df, x=param, y=metric_name, color='red',
                     estimator='mean', errorbar=None, ax=axs[i])
        axs[i].set_title(f'{param} vs {metric_name}')
    
    # Model Comparison (Middle Left)
    ax_model = fig.add_subplot(gs[1, 0:2])
    sns.boxplot(data=df, x='Model', y=metric_name, ax=ax_model)
    
    # Fix tick labels
    ax_model.set_xticklabels(ax_model.get_xticklabels(), rotation=45, ha='right')
    
    # Parameter Interactions Heatmap (Middle Right)
    ax_heat = fig.add_subplot(gs[1, 2:])
    try:
        best_idx = df[metric_name].idxmax() if higher_better else df[metric_name].idxmin()
        best_model = df.loc[best_idx, 'Model']
        
        best_df = df[df['Model'] == best_model].copy()
        if not best_df.empty:
            pivot_table = best_df.pivot_table(values=metric_name,
                                              index='padding_proportion',
                                              columns='probability_threshold',
                                              aggfunc='mean').fillna(0)
            sns.heatmap(pivot_table, annot=True, fmt=".2f", cmap='viridis', ax=ax_heat)
            ax_heat.set_title(f'Best Model ({best_model}) Parameter Interactions')
        else:
            ax_heat.axis('off')
            ax_heat.text(0.5, 0.5, 'No valid data for heatmap', ha='center', va='center')
    except Exception as e:
        ax_heat.axis('off')
        ax_heat.text(0.5, 0.5, f'Error creating heatmap: {str(e)}', ha='center', va='center')
    
    # Top Performers Table (Bottom Row)
    ax_table = fig.add_subplot(gs[2, :])
    ax_table.axis('off')
    try:
        top_params = df.nlargest(5, metric_name) if higher_better else df.nsmallest(5, metric_name)
        columns = ['Model'] + params + [metric_name]
        row_labels = [f"Top {i+1}" for i in range(min(5, len(top_params)))]
        
        if not top_params.empty:
            table_data = top_params[columns].values
            table_data_rounded = np.array([[str(cell) if i == 0 else round(float(cell), 3) 
                                            for i, cell in enumerate(row)] 
                                            for row in table_data])
            
            table = ax_table.table(cellText=table_data_rounded,
                                   colLabels=columns,
                                   rowLabels=row_labels,
                                   loc='center',
                                   cellLoc='center')
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1.2, 1.5)
        else:
            ax_table.text(0.5, 0.5, 'No valid data for table', ha='center', va='center')
    except Exception as e:
        ax_table.text(0.5, 0.5, f'Error creating table: {str(e)}', ha='center', va='center')

    plt.suptitle(f'Comprehensive Analysis: {metric_name}', y=0.98, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    # Save figure to the new folder
    save_path = os.path.join(save_folder, f"{metric_name}_dashboard.png")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close(fig)  # Close figure to free memory
    
    # Display saved figure
    display(Image(save_path))

# Identify all performance metrics
performance_metrics = {
    'total_recall': True,
    'total_precision': True,
    'overall_f1': True,
    'overall_mae': False,
    'overall_iou': True,
    'overall_score': True
}

# Generate dashboards
for metric, higher_better in performance_metrics.items():
    create_metric_dashboard(df, metric, higher_better)


In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Utility functions
# def plot_pie_chart(value, title, labels, colors=['#ff9999', '#66b3ff']):
#     """Generic function for plotting pie charts"""
#     explode = (0.05, 0)
#     fig, ax = plt.subplots()
#     wedges, texts, autotexts = ax.pie(
#         [value, 1-value],
#         explode=explode,
#         labels=labels,
#         autopct='%1.1f%%',
#         startangle=90,
#         colors=colors
#     )
#     ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
#     plt.title(title)
#     plt.show()

# def plot_pattern_pie_charts(pattern_metrics, metric_name, colors=['#ff9999', '#66b3ff']):
#     """Plot pie charts for each pattern"""
#     # Define labels and explode
#     labels = [f'Properly Located {metric_name}', f'Not Properly Located {metric_name}']
#     explode = (0.05, 0)
    
#     # Calculate number of patterns and subplot grid
#     num_patterns = len(pattern_metrics)
#     cols = min(7, num_patterns)
#     rows = (num_patterns + cols - 1) // cols
    
#     # Create subplots
#     fig, axs = plt.subplots(rows, cols, figsize=(cols*3.5, rows*4))
#     fig.suptitle(f'{metric_name} per Chart Pattern')
    
#     # Handle the axs array based on dimensions
#     if rows == 1 and cols == 1:
#         # Single plot case
#         wedges, texts, autotexts = axs.pie(
#             [list(pattern_metrics.values())[0], 1 - list(pattern_metrics.values())[0]],
#             explode=explode,
#             labels=[None, None],
#             autopct='%1.1f%%',
#             startangle=90,
#             colors=colors
#         )
#         axs.set_title(list(pattern_metrics.keys())[0])
#     else:
#         # Multiple plots case - need to handle 1D or 2D array
#         axs_flat = axs.flatten() if hasattr(axs, 'flatten') else [axs]
        
#         # Plot each pie chart
#         for i, (pattern, metric_value) in enumerate(pattern_metrics.items()):
#             if i < len(axs_flat):
#                 wedges, texts, autotexts = axs_flat[i].pie(
#                     [metric_value, 1 - metric_value],
#                     explode=explode,
#                     labels=[None, None],
#                     autopct='%1.1f%%',
#                     startangle=90,
#                     colors=colors
#                 )
#                 axs_flat[i].set_title(pattern)
                
#         # Hide unused subplots
#         for i in range(num_patterns, len(axs_flat)):
#             axs_flat[i].axis('off')
    
#     # Add a single legend for all pie charts
#     plt.figlegend(wedges, labels, loc="upper right")
#     plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for the title
#     plt.show()

# def plot_all_models_metrics_comparison(all_models_metrics, metric_name, metric_key):
#     """Plot bar chart comparing a specific metric across all models"""
#     plt.figure(figsize=(10, 6))
#     models = list(all_models_metrics.keys())
#     values = [metrics[metric_key] for metrics in all_models_metrics.values()]
    
#     colors = plt.cm.viridis(np.linspace(0, 0.8, len(models)))
#     plt.bar(models, values, color=colors)
#     plt.xlabel('Models')
#     plt.ylabel(metric_name)
#     plt.title(f'{metric_name} Comparison Across Models')
#     plt.xticks(rotation=45, ha='right')
#     plt.grid(axis='y', linestyle='--', alpha=0.7)
#     plt.tight_layout()
#     plt.show()

# def plot_metrics_bar_chart(metrics_dict, title, color='blue'):
#     """Plot metrics as a bar chart"""
#     plt.figure(figsize=(12, 6))
#     patterns = list(metrics_dict.keys())
#     values = list(metrics_dict.values())
    
#     plt.bar(patterns, values, color=color)
#     plt.xlabel('Chart Patterns')
#     plt.ylabel('Value')
#     plt.title(title)
#     plt.xticks(rotation=45, ha='right')
#     plt.grid(axis='y', linestyle='--', alpha=0.7)
#     plt.tight_layout()
#     plt.show()

# def plot_overall_metrics(overall_metrics_dict, title="Overall Model Metrics"):
#     """Plot overall metrics (F1, IoU, MAE) as a bar chart"""
#     plt.figure(figsize=(8, 6))
#     metrics = list(overall_metrics_dict.keys())
#     values = list(overall_metrics_dict.values())
    
#     colors = ['#4CAF50', '#2196F3', '#FFC107', '#FF5722']  # Green, Blue, Amber, Deep Orange
#     plt.bar(metrics, values, color=colors[:len(metrics)])
    
#     plt.xlabel('Metrics')
#     plt.ylabel('Value')
#     plt.title(title)
#     plt.grid(axis='y', linestyle='--', alpha=0.7)
#     plt.ylim(0, max(values) * 1.2)  # Add some headroom
    
#     # Add value labels on top of each bar
#     for i, v in enumerate(values):
#         plt.text(i, v + 0.02, f'{v:.4f}', ha='center')
        
#     plt.tight_layout()
#     plt.show()

# # Recall functions
# def calculate_per_pattern_recall(pattern_row_count, number_of_properly_located_patterns, test_patterns):
#     """Calculate recall for each pattern"""
#     per_pattern_recall = {}
#     for pattern, count in number_of_properly_located_patterns.items():
#         pattern_count = test_patterns[test_patterns['Chart Pattern'] == pattern].shape[0]
#         if pattern_count > 0:
#             per_pattern_recall[pattern] = count / pattern_count
#         else:
#             per_pattern_recall[pattern] = 0
#     return per_pattern_recall

# def get_and_plot_recall(pattern_row_count, number_of_properly_located_patterns, test_patterns):
#     """Calculate and plot overall recall and per-pattern recall"""
#     total_number_of_all_patterns = sum(pattern_row_count.values())
#     total_number_of_properly_located_patterns = sum(number_of_properly_located_patterns.values())
    
#     # Calculate total recall
#     total_recall = total_number_of_properly_located_patterns / total_number_of_all_patterns if total_number_of_all_patterns > 0 else 0
    
#     # Calculate per pattern recall
#     per_pattern_recall = calculate_per_pattern_recall(pattern_row_count, number_of_properly_located_patterns, test_patterns)
    
#     # Print recall metrics
#     print(f"Overall Recall: {total_recall:.4f}")
#     for pattern, recall in per_pattern_recall.items():
#         print(f"Recall for {pattern}: {recall:.4f}")
    
#     # Plot recall
#     plot_pie_chart(total_recall, "Overall Recall", 
#                   ['Properly Located Patterns', 'Not Properly Located Patterns'])
#     plot_pattern_pie_charts(per_pattern_recall, "Recall")
    
#     return total_recall, per_pattern_recall

# # Precision functions
# def calculate_per_pattern_precision(number_of_properly_located_patterns, located_patterns_df):
#     """Calculate precision for each pattern"""
#     per_pattern_precision = {}
#     for pattern, count in number_of_properly_located_patterns.items():
#         pattern_predictions = located_patterns_df[located_patterns_df['Chart Pattern'] == pattern].shape[0]
#         # Avoid division by zero
#         if pattern_predictions > 0:
#             per_pattern_precision[pattern] = count / pattern_predictions
#         else:
#             per_pattern_precision[pattern] = 0
#     return per_pattern_precision

# def get_and_plot_precision(pattern_row_count, number_of_properly_located_patterns, located_patterns_df):
#     """Calculate and plot overall precision and per-pattern precision"""
#     total_number_of_all_located_patterns = len(located_patterns_df)
#     total_number_of_properly_located_patterns = sum(number_of_properly_located_patterns.values())
    
#     # Avoid division by zero
#     if total_number_of_all_located_patterns > 0:
#         total_precision = total_number_of_properly_located_patterns / total_number_of_all_located_patterns
#     else:
#         total_precision = 0
    
#     # Calculate per pattern precision
#     per_pattern_precision = calculate_per_pattern_precision(number_of_properly_located_patterns, located_patterns_df)
    
#     # Print precision metrics
#     print(f"Overall Precision: {total_precision:.4f}")
#     for pattern, precision in per_pattern_precision.items():
#         print(f"Precision for {pattern}: {precision:.4f}")
    
#     # Plot precision
#     plot_pie_chart(total_precision, "Overall Precision", 
#                   ['Properly Located Patterns', 'All Located Patterns'], 
#                   colors=['lightgreen', 'skyblue'])
#     plot_pattern_pie_charts(per_pattern_precision, "Precision", colors=['lightgreen', 'skyblue'])
    
#     return total_precision, per_pattern_precision

# # F1 Score functions
# def calculate_f1_score(precision, recall):
#     """Calculate F1 score from precision and recall"""
#     if precision + recall == 0:
#         return 0
#     return 2 * (precision * recall) / (precision + recall)

# def calculate_per_pattern_f1(per_pattern_precision, per_pattern_recall):
#     """Calculate F1 score for each pattern"""
#     per_pattern_f1 = {}
#     for pattern in per_pattern_recall.keys():
#         precision = per_pattern_precision.get(pattern, 0)
#         recall = per_pattern_recall.get(pattern, 0)
#         if precision + recall > 0:
#             per_pattern_f1[pattern] = 2 * (precision * recall) / (precision + recall)
#         else:
#             per_pattern_f1[pattern] = 0
#     return per_pattern_f1

# def get_and_plot_f1(per_pattern_precision, per_pattern_recall):
#     """Calculate and display F1 scores"""
#     per_pattern_f1 = calculate_per_pattern_f1(per_pattern_precision, per_pattern_recall)
    
#     # Calculate overall F1 score
#     all_precisions = list(per_pattern_precision.values())
#     all_recalls = list(per_pattern_recall.values())
#     avg_precision = sum(all_precisions) / len(all_precisions) if all_precisions else 0
#     avg_recall = sum(all_recalls) / len(all_recalls) if all_recalls else 0
#     overall_f1 = calculate_f1_score(avg_precision, avg_recall)
    
#     # Print F1 scores
#     print(f"Overall F1 Score: {overall_f1:.4f}")
#     for pattern, f1 in per_pattern_f1.items():
#         print(f"F1 Score for {pattern}: {f1:.4f}")
    
#     # Plot F1 scores as bar chart
#     plot_metrics_bar_chart(per_pattern_f1, "F1 Score by Pattern", color='orange')
    
#     return overall_f1, per_pattern_f1

# # MAE functions
# def calculate_per_pattern_mae(mae_for_each_properly_detected_pattern, number_of_properly_located_patterns):
#     """Calculate Mean Absolute Error for each pattern"""
#     per_pattern_mae = {}
#     for pattern, count in number_of_properly_located_patterns.items():
#         if count > 0:
#             per_pattern_mae[pattern] = mae_for_each_properly_detected_pattern.get(pattern, 0) / count
#         else:
#             per_pattern_mae[pattern] = 0
#     return per_pattern_mae

# def get_and_plot_mae(mae_for_each_properly_detected_pattern, number_of_properly_located_patterns):
#     """Calculate and display MAE metrics"""
#     per_pattern_mae = calculate_per_pattern_mae(mae_for_each_properly_detected_pattern, number_of_properly_located_patterns)
    
#     total_mae_sum = sum(mae_for_each_properly_detected_pattern.values())
#     total_proper_patterns = sum(number_of_properly_located_patterns.values())
#     overall_mae = total_mae_sum / total_proper_patterns if total_proper_patterns > 0 else 0
    
#     # Print MAE metrics
#     print(f"Overall Mean Absolute Error: {overall_mae:.4f}")
#     for pattern, mae in per_pattern_mae.items():
#         print(f"Mean Absolute Error for {pattern}: {mae:.4f}")
    
#     # Plot MAE as bar chart
#     plot_metrics_bar_chart(per_pattern_mae, "Mean Absolute Error by Pattern", color='red')
    
#     return overall_mae, per_pattern_mae

# # IoU functions
# def calculate_per_pattern_iou(iou_for_each_properly_detected_pattern, number_of_properly_located_patterns):
#     """Calculate Mean Intersection over Union for each pattern"""
#     per_pattern_iou = {}
#     for pattern, count in number_of_properly_located_patterns.items():
#         if count > 0:
#             per_pattern_iou[pattern] = iou_for_each_properly_detected_pattern.get(pattern, 0) / count
#         else:
#             per_pattern_iou[pattern] = 0
#     return per_pattern_iou

# def get_and_plot_iou(iou_for_each_properly_detected_pattern, number_of_properly_located_patterns):
#     """Calculate and display IoU metrics"""
#     per_pattern_iou = calculate_per_pattern_iou(iou_for_each_properly_detected_pattern, number_of_properly_located_patterns)
    
#     total_iou_sum = sum(iou_for_each_properly_detected_pattern.values())
#     total_proper_patterns = sum(number_of_properly_located_patterns.values())
#     overall_iou = total_iou_sum / total_proper_patterns if total_proper_patterns > 0 else 0
    
#     # Print IoU metrics
#     print(f"Overall Mean Intersection over Union: {overall_iou:.4f}")
#     for pattern, iou in per_pattern_iou.items():
#         print(f"Mean Intersection over Union for {pattern}: {iou:.4f}")
    
#     # Plot IoU metrics - now using bar chart for both
#     plot_metrics_bar_chart(per_pattern_iou, "Mean IoU by Pattern", color='green')
    
#     return overall_iou, per_pattern_iou

# # Visualization functions
# def plot_probability_distribution(probabilities, threshold, pattern_encoding_reversed):
#     """Plot probability distribution with threshold highlighting"""
#     indices = np.arange(len(probabilities))
#     colors = ["red" if p >= threshold else "blue" for p in probabilities]
#     pattern_labels = [f"{pattern_encoding_reversed[i]}" for i in range(len(probabilities))]
    
#     plt.figure(figsize=(10, 5))
#     plt.bar(indices, probabilities, color=colors)
#     plt.axhline(y=threshold, color="gray", linestyle="--", label=f"Threshold = {threshold}")
#     plt.xticks(indices, pattern_labels, rotation=45, ha='right')
#     plt.xlabel("Pattern Type")
#     plt.ylabel("Probability")
#     plt.title("Probability Distribution with Threshold Highlighting")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

# def evaluate_model(model_name, model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_dict):
#     """Evaluate a model and display all metrics"""
#     print(f"\n{'='*20} Model: {model_name} {'='*20}")
    
#     # Extract model results
#     number_of_properly_located_patterns = model_eval_results_dict[model_name]['number_of_properly_located_patterns']
#     located_patterns_df = located_patterns_dict[model_name]
#     mae_for_each_properly_detected_pattern = model_eval_results_dict[model_name]['mae_for_each_properly_detected_pattern']
#     iou_for_each_properly_detected_pattern = model_eval_results_dict[model_name]['iou_for_each_properly_detected_pattern']
    
#     # Calculate and plot metrics
#     print("\n--- Recall Metrics ---")
#     total_recall, per_pattern_recall = get_and_plot_recall(pattern_row_count, number_of_properly_located_patterns, test_patterns)
    
#     print("\n--- Precision Metrics ---")
#     total_precision, per_pattern_precision = get_and_plot_precision(pattern_row_count, number_of_properly_located_patterns, located_patterns_df)
    
#     print("\n--- F1 Score Metrics ---")
#     overall_f1, per_pattern_f1 = get_and_plot_f1(per_pattern_precision, per_pattern_recall)
    
#     print("\n--- Mean Absolute Error Metrics ---")
#     overall_mae, per_pattern_mae = get_and_plot_mae(mae_for_each_properly_detected_pattern, number_of_properly_located_patterns)
    
#     print("\n--- Intersection over Union Metrics ---")
#     overall_iou, per_pattern_iou = get_and_plot_iou(iou_for_each_properly_detected_pattern, number_of_properly_located_patterns)
    
#     # Plot overall metrics on a single chart
#     overall_metrics = {
#         'Recall': total_recall,
#         'Precision': total_precision,
#         'F1 Score': overall_f1,
#         'IoU': overall_iou,
#         'MAE': overall_mae
#     }
#     plot_overall_metrics(overall_metrics, f"Overall Metrics for {model_name}")
    
#     # Store all metrics in one place for easy access
#     metrics_summary = {
#         'total_recall': total_recall,
#         'per_pattern_recall': per_pattern_recall,
#         'total_precision': total_precision,
#         'per_pattern_precision': per_pattern_precision,
#         'overall_f1': overall_f1,
#         'per_pattern_f1': per_pattern_f1,
#         'overall_mae': overall_mae,
#         'per_pattern_mae': per_pattern_mae,
#         'overall_iou': overall_iou,
#         'per_pattern_iou': per_pattern_iou
#     }
    
#     return metrics_summary

# # Main function to evaluate all models
# def evaluate_all_models(model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_dict):
#     """Evaluate all models and return metrics summary"""
#     all_models_metrics = {}
    
#     for model_name in model_eval_results_dict.keys():
#         all_models_metrics[model_name] = evaluate_model(
#             model_name,
#             model_eval_results_dict,
#             pattern_row_count,
#             test_patterns,
#             located_patterns_dict
#         )
    
#     # After evaluating all models, compare their overall metrics
#     if len(model_eval_results_dict) > 1:
#         print("\n--- Comparing Models ---")
        
#         # Compare models on key metrics
#         plot_all_models_metrics_comparison(all_models_metrics, "Recall", "total_recall")
#         plot_all_models_metrics_comparison(all_models_metrics, "Precision", "total_precision")
#         plot_all_models_metrics_comparison(all_models_metrics, "F1 Score", "overall_f1")
#         plot_all_models_metrics_comparison(all_models_metrics, "Mean IoU", "overall_iou")
#         plot_all_models_metrics_comparison(all_models_metrics, "Mean Absolute Error", "overall_mae")
    
#     return all_models_metrics

# # Example of how to use the functions:
# all_metrics = evaluate_all_models(model_eval_results_dict, pattern_row_count, test_patterns, located_patterns_and_other_info_updated_dict)